## forced photometry and multi-band periodogram Hello World
**David W. Hogg** (NYU) (MPIA)

### acknowledgement
- I made use of *Claude* for advice and minimal code and documentation help.

### projects and to-dos
- Compute a goodness of fit or likelihood at the end of IRLS and rank peaks by this statistic.
- Need to test / verify the barycentric correction.
- Switch from RA, Dec as input to object id.
- Figure out how to iterate over all possible object ids.
- Make the period fitting fast.

### RSP token notes
*The code below needs an RSP token. Here are Claude's notes about this:*
1. Go to data.lsst.cloud in a browser and log in.
1. Open the user drop-down at the upper right and choose Settings.
1. In the left menu, choose Access tokens.
1. Click Create a token.
1. In the dialog:
1. Name: something like pyvo-tap, named after what you’ll use it with.
1. Scopes: tick read:tap. For image retrieval later, also add read:image.
1. Expiration: set a date, or choose no expiration.
1. Click Create token. The token is shown once, so copy it right away. If you lose it, delete it and make a new one.
2. Save the token with `echo 'RSP_TOKEN=paste-token-here' > .env`

In [ ]:
# !pip install pyvo
# !pip install python-dotenv
# !pip install "lsst-utils[plotting]"

In [ ]:
import os
from dotenv import load_dotenv
import numpy as np
import matplotlib.pyplot as plt
import pyvo, requests
from scipy.signal import find_peaks
from lsst.utils.plotting import get_multiband_plot_colors
from lsst.utils.plotting import get_multiband_plot_symbols
import astropy.units as u
from astropy.time import Time
from astropy.coordinates import SkyCoord, EarthLocation

In [ ]:
colors = get_multiband_plot_colors()
markers = get_multiband_plot_symbols()

In [ ]:
# step 0: authenticate and connect
load_dotenv()
rsp_token = os.environ["RSP_TOKEN"]
session = requests.Session()
session.headers["Authorization"] = f"Bearer {rsp_token}"
tap = pyvo.dal.TAPService("https://data.lsst.cloud/api/tap", session=session)

In [ ]:
def run(query):
    job = tap.submit_job(query)
    job.run()
    job.wait(phases=["COMPLETED", "ERROR"])
    job.raise_if_error()
    return job.fetch_result().to_table()   # astropy Table

In [ ]:
# step 1: find the objectId at a sky position
ra, dec = 226.7244283498404, -40.30405425755786 # known RR Lyrae
obj = run(f"""
    SELECT objectId, coord_ra, coord_dec
    FROM dp2.Object
    WHERE CONTAINS(POINT('ICRS', coord_ra, coord_dec),
                   CIRCLE('ICRS', {ra}, {dec}, 0.00056)) = 1
""")
object_id = int(obj["objectId"][0])      # expect 743582465276261182

In [ ]:
# step 2: forced photometry for that object, joined to Visit for time
lc = run(f"""
    SELECT fs.band, fs.visit, fs.psfFlux, fs.psfFluxErr,
           fs.psfDiffFlux, fs.psfDiffFluxErr, v.expMidptMJD
    FROM dp2.ForcedSource AS fs
    JOIN dp2.Visit AS v ON fs.visit = v.visit
    WHERE fs.objectId = {object_id}
""")
print(len(lc), "epochs")                # the tutorial reports 887

In [ ]:
# step 2.5: make barycentric corrections
location = EarthLocation.of_site("Rubin Observatory")
ttt = Time(lc["expMidptMJD"], format='mjd', scale='utc', location=location)
target = SkyCoord(ra=obj["coord_ra"], dec=obj["coord_dec"], frame='icrs')
ltt_bary = ttt.light_travel_time(target, kind='barycentric')
bjd_tdb = (ttt.tdb + ltt_bary).jd
bjd_tdb0 = np.round(np.median(bjd_tdb))
print(bjd_tdb0)

In [ ]:
# step 3: plot (fluxes in nJy)
fig, ax = plt.subplots(figsize=(7, 4))
for band in "ugrizy":
    m = lc["band"] == band
    if m.any():
        ax.errorbar(bjd_tdb[m] - bjd_tdb0, lc["psfFlux"][m],
                    yerr=lc["psfFluxErr"][m], fmt=markers[band], ms=3.,
                    color=colors[band], label=band, alpha=0.7)
ax.set_xlabel(f"BJD TDB - {bjd_tdb0:.0f} d")
ax.set_ylabel("psfFlux [nJy]")
ax.legend(ncol=3)
plt.show()

In [ ]:
def one_band_design_matrix(om, M, t):
    """
    ## bugs:
    - Doesn't use `jax-finufft`?
    - Copied verbatim from `KeplerClocks`.
    """
    ms1, ms2 = np.arange(M + 1), np.arange(1, M + 1)
    return np.concat((np.cos(ms1[None, :] * om * t[:, None]),
                      np.sin(ms2[None, :] * om * t[:, None])), axis=1), \
           np.concat((ms1, ms2))

def multi_band_design_matrix(om, M, ts, bs):
    """
    ## inputs:
    - `om`: Fundamental angular frequency
    - `M`:  Degree of the Fourier series
    - `ts`: List of times, one per observation
    - `bs`: List of bands, one per observation

    ## bugs:
    - Uses `one_band_design_matrix()`.
    """
    assert len(ts) == len(bs)
    bands = np.sort(np.unique(bs))
    parameter_bands = np.array([[b, ] * (2 * M + 1) for b in bands]).flatten()
    parameter_ms = np.zeros(len(parameter_bands)).astype(int)
    design_matrix = np.zeros((len(ts), len(parameter_bands)))
    for b in bands:
        obschunk = (bs == b)
        parschunk = (parameter_bands == b)
        design_matrix[np.ix_(obschunk, parschunk)], parameter_ms[parschunk] = one_band_design_matrix(om, M, ts[obschunk])
    return design_matrix, parameter_ms, parameter_bands

In [ ]:
# test design matrix code
X, ms, bs  = multi_band_design_matrix(1.0, 2, np.array(lc["expMidptMJD"]), np.array(lc["band"]))
print(X.shape, np.all(np.isfinite(X)))
print(ms, bs)

In [ ]:
# step 4: set up data for likelihood functions
MJD0 = np.round(np.median(lc["expMidptMJD"]))
ts = bjd_tdb - bjd_tdb0
bs = np.array(lc["band"])
ys = np.array(lc["psfFlux"])
ivars = 1. / np.array(lc["psfFluxErr"]) ** 2

In [ ]:
def one_frequency_chisq(f, M, ts, bs, ys, ws, return_pars=False):
    X, _, _ = multi_band_design_matrix(2. * np.pi * f, M, ts, bs)
    XTX = X.T @ (ws[:, None] * X)
    XTy = X.T @ (ws * ys)
    pars = np.linalg.solve(XTX, XTy)
    resids = ys - X @ pars
    if return_pars:
        return np.sum(ws * resids * resids), X, pars
    return np.sum(ws * resids * resids)

In [ ]:
# step 5: perform slow periodogram
df = 1. / (np.max(ts) - np.min(ts))
fmin = 1. / 2. # inverse 2 days
fmax = 1. / 0.25 # inverse 0.5 days
fs = np.arange(fmin, fmax, 0.25 * df)
chis = np.zeros_like(fs)
M = 2
ofc = lambda f: one_frequency_chisq(f, M, ts, bs, ys, ivars)
chi2s = np.array(list(map(ofc, fs)))
print(ys.shape, fs.shape, chi2s.shape)

In [ ]:
# find peaks
max_peaks = 5
idxs, _ = find_peaks(-0.5 * chi2s, distance=4) # frequency squared                       
ii = np.argsort(chi2s[idxs])
idxs = idxs[ii]
if len(idxs) > max_peaks:
    idxs = idxs[:max_peaks]
print("top frequencies:", fs[idxs])

In [ ]:
def refine_peak_irls(f, M, ts, bs, ys, input_ws, maxiter=10, Q=3.):
    ws = 1. * input_ws
    for i in range(maxiter):
        chi2, X, pars = one_frequency_chisq(f, M, ts, bs, ys, ws, return_pars=True)
        resids = ys - X @ pars
        ws = input_ws * Q * Q / (Q * Q + input_ws * resids * resids)
    return pars, ws, resids

In [ ]:
# step 6: refine peaks with IRLS and high M; plot data folded at key frequencies

M = 5 # go to higher M for refinement
thetaplot = np.linspace(0., 4. * np.pi, 1000)
for i in idxs:
    fig, ax = plt.subplots(figsize=(7, 4))
    for band in "ugrizy":
        m = (bs == band)
        if m.any():
            pars, _, _ = refine_peak_irls(fs[i], M, ts[m], bs[m], ys[m], ivars[m])
            Xplot, _ = one_band_design_matrix(2. * np.pi * fs[i], M, thetaplot / (2. * np.pi * fs[i]))
            plt.plot(thetaplot, Xplot @ pars, linestyle="-", color=colors[band], lw=1, alpha=0.5)
            ax.errorbar((fs[i] * ts[m] % 1.) * 2. * np.pi, ys[m],
                        yerr = 1. / np.sqrt(ivars[m]), fmt=markers[band], ms=3.,
                        color=colors[band], label=band, alpha=0.7)
            ax.errorbar((fs[i] * ts[m] % 1.) * 2. * np.pi + 2. * np.pi, ys[m],
                        yerr = 1. / np.sqrt(ivars[m]), fmt=markers[band], ms=3.,
                        color=colors[band], alpha=0.7)
    ax.set_xlabel("phase (rad)"); ax.set_ylabel("psfFlux [nJy]"); ax.legend(ncol=3)
    ax.set_title(f"data folded at a period of {1. / fs[i]:.3f} d")
    ax.set_xlim(0., 4. * np.pi)
    ax.set_ylim(0., 2.5 * np.median(ys))
    plt.show()